In [1]:
### type your query here ###
query = "Generate a podcast content between two people discussing their favorite class of computer science."

In [2]:
import sys
import os
sys.path.append(os.path.abspath("../"))
from rag.retrieval import retrieve

results = retrieve(query)

/Users/coconut/Coding101/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
results

[{'source': 'chap_01.txt',
  'category': 'Computer',
  'chunk_index': 2,
  'page': 3,
  'text': 'Classes of Computers (2/2) Supercomputers High-end scientific and engineering calculations Highest capability but represent a small fraction of the overall computer market Embedded computers Hidden as components of systems ex: washing machines, cars, cell phones, PDA, … Stringent power/performance/cost constraints  The PostPC Era (1/2)'},
 {'source': 'chap_01.txt',
  'category': 'Computer',
  'chunk_index': 2,
  'page': 3,
  'text': 'Classes of Computers (2/2) Supercomputers High-end scientific and engineering calculations Highest capability but represent a small fraction of the overall computer market Embedded computers Hidden as components of systems ex: washing machines, cars, cell phones, PDA, … Stringent power/performance/cost constraints  The PostPC Era (1/2)'},
 {'source': 'chap_01.txt',
  'category': 'Computer',
  'chunk_index': 2,
  'page': 3,
  'text': 'Classes of Computers (2/2) 

In [4]:
from openai import OpenAI
import os

# 初始化 OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def generate_dialog(user_instruction, retrieved_context, model="gpt-4o-mini", max_tokens=512):
    """使用 OpenAI API 生成對話"""
    prompt = f"""Use the information provided in the relevant context to generate a natural dialogue between two people (Person A and Person B).

### Relevant Context:
{retrieved_context}

### Instruction:
{user_instruction}

### Requirements:
- Format each line as "A: [text]" or "B: [text]"
- Make it conversational and natural
- Keep responses focused on the context provided

### Dialogue:"""
    
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant that generates natural dialogues between two people based on given context."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=0.8,
        top_p=0.9
    )
    
    return response.choices[0].message.content.strip()

# 使用檢索到的上下文生成對話
retrieved_texts = "\n\n".join([r['text'] for r in results[:3]])  # 使用前3個結果
llm_output = generate_dialog(query, retrieved_texts)
print(llm_output)

A: Hey there, welcome back to our tech podcast! Today, I thought we could dive into our favorite classes of computers. What do you think?

B: That sounds great! I’ve always been fascinated by supercomputers. Their ability to handle high-end scientific and engineering calculations is just incredible. 

A: Supercomputers are definitely impressive! But I think I’m more drawn to embedded computers. They’re so interesting because they’re hidden within everyday devices like washing machines and cars.

B: That's true! Embedded computers really do play a crucial role in our daily lives without us even realizing it. They have such stringent power and performance constraints, which makes their design quite challenging.

A: Exactly! It’s amazing how something so small can control so much. Plus, with the rise of the PostPC era, embedded systems are becoming more prevalent.

B: Yes! The PostPC era is fascinating. It’s like we’re moving away from traditional computing devices to these smaller, embed

In [5]:
import re

def parse_dialog_to_kokoro(dialog_text):
    lines = dialog_text.strip().split('\n')
    result = []
    for line in lines:
        m = re.match(r'^([AB]):\s*(.*)', line.strip())
        if m:
            speaker, text = m.groups()
            result.append((speaker, text))
    return result

kokoro_dialog = parse_dialog_to_kokoro(llm_output)
print(kokoro_dialog)

[('A', 'Hey there, welcome back to our tech podcast! Today, I thought we could dive into our favorite classes of computers. What do you think?'), ('B', 'That sounds great! I’ve always been fascinated by supercomputers. Their ability to handle high-end scientific and engineering calculations is just incredible.'), ('A', 'Supercomputers are definitely impressive! But I think I’m more drawn to embedded computers. They’re so interesting because they’re hidden within everyday devices like washing machines and cars.'), ('B', "That's true! Embedded computers really do play a crucial role in our daily lives without us even realizing it. They have such stringent power and performance constraints, which makes their design quite challenging."), ('A', 'Exactly! It’s amazing how something so small can control so much. Plus, with the rise of the PostPC era, embedded systems are becoming more prevalent.'), ('B', 'Yes! The PostPC era is fascinating. It’s like we’re moving away from traditional computi

In [ ]:
# !pip install -q kokoro>=0.9.2 soundfile
# !apt-get -qq -y install espeak-ng > /dev/null 2>&1
from kokoro import KPipeline
from IPython.display import display, Audio
import soundfile as sf
import torch

: 

In [ ]:
pipeline = KPipeline(lang_code='a')

VOICE_MAP = {
    "A": "af_heart",
    "B": "am_adam",
}

# Clear any previous outputs to free memory
from IPython.display import clear_output

for i, (speaker, text) in enumerate(kokoro_dialog):
    print(f"\n[{i}] {speaker}: {text}")
    
    try:
        # 生成音频
        generator = pipeline(text, voice=VOICE_MAP[speaker])
        
        for _, _, audio in generator:
            # 保存音频文件
            outfile = f"{i:02d}_{speaker}.wav"
            sf.write(outfile, audio, 24000)
            print(f"✓ 已保存: {outfile}")
            
            # 显示音频播放器（不自动播放）
            display(Audio(data=audio, rate=24000, autoplay=False))
            break  # 只取第一个结果
            
    except Exception as e:
        print(f"✗ 生成失败: {e}")
        continue

print("\n所有音频生成完成！")


In [9]:
from pydub import AudioSegment
import os

def merge_wavs(wav_files, output="final.wav"):
    audio = AudioSegment.empty()

    for wav in wav_files:
        print("Adding:", wav)
        segment = AudioSegment.from_wav(wav)
        audio += segment

    audio.export(output, format="wav")
    print("Saved merged file:", output)

wav_files = sorted([f for f in os.listdir(".") if f.endswith(".wav")])
merge_wavs(wav_files, "podcast_dialogue.wav")

Adding: 00_A.wav
Adding: 01_B.wav
Adding: 02_A.wav
Adding: 03_B.wav
Adding: 04_A.wav
Adding: 05_B.wav
Adding: 06_A.wav
Adding: 07_B.wav
Adding: 08_A.wav
Adding: 09_B.wav
Adding: 10_A.wav
Saved merged file: podcast_dialogue.wav
